# Import libries

In [2]:

# %% [markdown]
# # Final Project: Customer Segmentation & Churn Prediction System
# **Diyaariye:** Muhiadin Said Hassan
# **Ujeeddada:** In macaamiisha la kooxeeyo (K-Means) ka dibna la saadaaliyo kuwa bixi raba (Supervised Classification).

# %%
import os
# Deji dunaha processor-ka si looga hortago Memory Leak-ga Windows-ka marka K-Means la isticmaalayo
os.environ["OMP_NUM_THREADS"] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [3]:
warnings.filterwarnings('ignore')

# 1. SOO DEJINTA XOGTA (LOAD DATASET)

In [4]:

print("=== Talaabada 1-aad: Soo dejinta Xogta ===")
# Xogta Telco Churn ee rasmiga ah (Waxaad ka soo dejisan kartaa Kaggle ama link-ga IBM)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

print(f"Xogta la soo dejiyay waxay ka kooban tahay: {df.shape[0]} saf iyo {df.shape[1]} tiro.")
print("\n--- Shanta saf ee ugu horreysa xogta ---")
print(df.head())

=== Talaabada 1-aad: Soo dejinta Xogta ===
Xogta la soo dejiyay waxay ka kooban tahay: 7043 saf iyo 21 tiro.

--- Shanta saf ee ugu horreysa xogta ---
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...         

# 2. NADIIFINTA IYO DIYAARINTA XOGTA (PREPROCESSING)

In [5]:

print("\n=== Talaabada 2-aad: Nadiifinta Xogta ===")

# Ka saar tiirka CustomerID madaama uusan muhiim u ahayn barashada model-ka
df.drop(columns=['customerID'], errors='ignore', inplace=True)

# TotalCharges badanaa qoraal (object) ayay ahaanaysaa, waxaan u beddeleynaa lambar (numeric)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')

# Meelaha eberka (null values) ku jiraan waxaan ku buuxineynaa Median-ka tiirkaas
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# U beddel tiirka Target-ka (Churn) lambarro: Yes = 1, No = 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print("Nadiifinta waa dhammaatay. Ma jiraan meelo eber ah:", df.isnull().sum().sum())


=== Talaabada 2-aad: Nadiifinta Xogta ===
Nadiifinta waa dhammaatay. Ma jiraan meelo eber ah: 0


# 3. KOOXAYNTA MACAAMIISHA (UNSUPERVISED LEARNING - K-MEANS)

In [6]:

print("\n=== Talaabada 3-aad: Kooxaynta Macaamiisha (K-Means) ===")

# Waxaan kooxaynta u isticmaalaynaa saddexda tiir ee lacagta iyo mudada (Continuous features)
cluster_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
X_cluster = df[cluster_features]

# Qiyaasidda xogta (Scaling) ka hor intaanan K-Means ku rulin
scaler_geo = StandardScaler()
X_cluster_scaled = scaler_geo.fit_transform(X_cluster)

# Helidda inta kooxood ee ugu habboon (Elbow Method)
sse = []
for k in range(1, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X_cluster_scaled)
    sse.append(km.inertia_)

# Sawiridda Shaxda Elbow Method (Waa ikhtiyaari laakiin muhiim ayay u tahay warbixinta)
plt.figure(figsize=(6, 4))
plt.plot(range(1, 7), sse, marker='o', color='purple')
plt.title('Elbow Method - Doorashada K')
plt.xlabel('Tirada Kooxaha (K)')
plt.ylabel('SSE / Inertia')
plt.grid(True)
plt.savefig('elbow_telecom.png')
plt.close()
print("[Checkpoint] Shaxda Elbow Method waxaa loo keydiyay 'elbow_telecom.png'.")

# Waxaan dooranay K=3 (Macaamiisha waxaan u kala qeybinaynaa 3 kooxood)
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
df['Segment_ID'] = kmeans.fit_predict(X_cluster_scaled)

print("\nKala qeybsanka macaamiisha ee 3-da kooxood:")
print(df['Segment_ID'].value_counts())


=== Talaabada 3-aad: Kooxaynta Macaamiisha (K-Means) ===
[Checkpoint] Shaxda Elbow Method waxaa loo keydiyay 'elbow_telecom.png'.

Kala qeybsanka macaamiisha ee 3-da kooxood:
Segment_ID
2    2689
1    2200
0    2154
Name: count, dtype: int64


# 4. DIYAARINTA MODEL-KA SAADAALINTA (SUPERVISED PREPARATION)

In [7]:
print("\n=== Talaabada 4-aad: Diyaarinta Model-ka Saadaalinta ===")

# Segment_ID oo ah kooxayntii K-Means waxaan ku darnay xogta guud si uu model-ka u barto
X = df.drop(columns=['Churn'])
y = df['Churn']

# Kala bixi xogta (Train/Test Split - 80% Tareen iyo 20% Tijaabo)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Kala saar tiirarka qoraalka ah (Categorical) iyo kuwa lambarka ah (Numerical)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [col for col in X.columns if col not in num_cols and col != 'Segment_ID']

# Samee dhuumaha beddelka xogta (Pipeline transformation)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ], remainder='passthrough')

# Beddel xogta tababarka iyo tijaabada
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Xogta si guul leh ayaa loo beddelay. Habka loo diyaariyay qaab matrix ah.")


=== Talaabada 4-aad: Diyaarinta Model-ka Saadaalinta ===
Xogta si guul leh ayaa loo beddelay. Habka loo diyaariyay qaab matrix ah.


# 5. TABABARIDDA IYO QIIMAYNTA MODEL-KADA (TRAINING & EVALUATION)

In [8]:

print("\n=== Talaabada 5-aad: Tababaridda iyo Qiimaynta Model-kada ===")

# Diyaarinta labada Model: Logistic Regression iyo Random Forest
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100)
}

# Loop lagu tababarayo laguna qiimaynayo model kasta iyadoo xoogga la saarayo F1-Score
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred = model.predict(X_test_processed)
    
    print(f"\n--- Natiijada Model-ka: {name} ---")
    print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
    print(f"F1-Score : {f1_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))


=== Talaabada 5-aad: Tababaridda iyo Qiimaynta Model-kada ===

--- Natiijada Model-ka: Logistic Regression ---
Accuracy : 0.8062
Precision: 0.6593
Recall   : 0.5588
F1-Score : 0.6049

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409


--- Natiijada Model-ka: Random Forest ---
Accuracy : 0.7885
Precision: 0.6293
Recall   : 0.4947
F1-Score : 0.5539

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409


# 6. KEYDINTA NATIIJADA FINAL-KA AH

In [10]:
print("\n=== Talaabada 6-aad: Keydinta Xogta Macaamiisha ee la Kooxeeyay ===")
df.to_csv('telecom_segmented_customers.csv', index=False)
print("Xogta final-ka ah ee wadata kooxaha (Segment_ID) waxaa lagu keydiyay 'telecom_segmented_customers")


=== Talaabada 6-aad: Keydinta Xogta Macaamiisha ee la Kooxeeyay ===
Xogta final-ka ah ee wadata kooxaha (Segment_ID) waxaa lagu keydiyay 'telecom_segmented_customers
